<a href="https://colab.research.google.com/github/tusharxtech/Hotel-Attribute-Extraction-Tiering/blob/main/Hotel_Attribute_Extraction_%26_Tiering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install the kaggle to take data from it**

**connect it by kaggle api key**

In [2]:
!pip install kaggle
import os
import json

kaggle_config = {
    "username": "tusharsinghlodhi",
    "key": "KGAT_f0dc9d58b843e5efe1c9cd9a9d4d51d3"
}

os.makedirs('/root/.kaggle', exist_ok=True)

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_config, f)

!chmod 600 /root/.kaggle/kaggle.json

**Download data**

In [3]:
!kaggle datasets download -d andrewmvd/trip-advisor-hotel-reviews

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/trip-advisor-hotel-reviews
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 5.14M/5.14M [00:00<00:00, 70.8MB/s]



**Unxip the data into csv file**

In [4]:
import zipfile

with zipfile.ZipFile('/content/trip-advisor-hotel-reviews.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

**looking data**

In [5]:
import pandas as pd
df=pd.read_csv('/content/data/tripadvisor_hotel_reviews.csv')
print(df.shape)
df.head()

(20491, 2)


,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5


dropping na if present

In [6]:
print(df.isnull().sum())


Review    0
Rating    0
dtype: int64


**checking for imbalance data**

In [7]:
count_5 = df[df['Rating'] == 5].shape[0]
print("count_5",count_5)
count_4 = df[df['Rating'] == 4].shape[0]
print("count_4",count_4)
count_3 = df[df['Rating'] == 3].shape[0]
print("count_3",count_3)
count_2 = df[df['Rating'] == 2].shape[0]
print("count_2",count_2)
count_1 = df[df['Rating'] == 1].shape[0]
print("count_1",count_1)

count_5 9054
count_4 6039
count_3 2184
count_2 1793
count_1 1421


In [8]:
df.head()
df.shape

(20491, 2)

down sample
because too many data and data imbalance

In [9]:
from sklearn.utils import resample

# Find the size of the smallest class (Rating 1)
min_size = df['Rating'].value_counts().min()

# Downsample each group
balanced_df = pd.concat([
    resample(df[df['Rating'] == i], replace=False, n_samples=150, random_state=42)
    for i in df['Rating'].unique()
])

print(balanced_df['Rating'].value_counts())

Rating
4    150
2    150
3    150
5    150
1    150
Name: count, dtype: int64


In [10]:
print(balanced_df.shape)
balanced_df.head()

(750, 2)


,Review,Rating
18754,excellent allround traffic noise junior hotel ...,4
2440,"magnificent trip vieques, love staying somewha...",4
16540,tried tested formula aman really got pat fabul...,4
7256,great setting fully renovated bit expensive bo...,4
4232,touring toronto stayed friends day weekend vis...,4


In [11]:
from collections import defaultdict
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


splitting sentences on punctuations

In [12]:
def split_into_sentences(text):
    """Split review text into individual sentences."""
    try:
        sentences = sent_tokenize(str(text))
        # Filter very short sentences (noise)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 15]
        return sentences
    except:
        return [str(text)]

balanced_df['sentences'] = balanced_df['Review'].apply(split_into_sentences)


print(f"Total sentences: {balanced_df['sentences'].explode().shape[0]}")


Total sentences: 762


In [13]:
balanced_df.head()

,Review,Rating,sentences
18754,excellent allround traffic noise junior hotel ...,4,[excellent allround traffic noise junior hotel...
2440,"magnificent trip vieques, love staying somewha...",4,"[magnificent trip vieques, love staying somewh..."
16540,tried tested formula aman really got pat fabul...,4,[tried tested formula aman really got pat fabu...
7256,great setting fully renovated bit expensive bo...,4,[great setting fully renovated bit expensive b...
4232,touring toronto stayed friends day weekend vis...,4,[touring toronto stayed friends day weekend vi...


using mistral as it is opensource

In [14]:
!pip install mistralai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.0/996.0 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.38.0
    Uninstalling opentelemetry-api-1.38.0:
      Successfully uninstalled opentelemetry-api-1.38.0
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.59b0
    Uninstalling opentelemetry-semantic-conventions-0.59b0:
      Successfully uninstalled opentelemetry-semantic-conventions-0.59b0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetr

# **labeling strategy ---- LLM Labeling Function ----**

used LLM-based labeling because the dataset has no attribute-level ground truth, and LLMs can generate high-quality, structured labels quickly without manual annotation.

and Captures complex language

Good tradeoff for this assignment
Time constraint

used 5 attributes:
1. cleanliness - how clean the rooms, bathrooms, common areas are
2. staff_service - quality of staff, helpfulness, friendliness
3. wifi_quality - internet/wifi speed and reliability
4. location_accessibility - proximity to public transit
5. safety_&_securiy - safety on rooms and for luggage

the factors i considered to be the most common

used batched processing, rate_limiting, retry and delay for llm to be safe and does not return error 429

provide "labeled_reviews.csv" which is data with different attribute later to be computed for tiering the hotel and aggregating accordingly

In [15]:
from mistralai.client import Mistral
import re, time, json
import pandas as pd
from tqdm import tqdm

client = Mistral(api_key="sAMJqlVx1c5ibTq3ImJRfiWb1I9bt8FW")
model = "mistral-large-latest"

ATTRIBUTES = ["cleanliness", "staff_service", "wifi_quality", "location_accessibility", "safety_&_securiy"]

BATCH_SIZE = 10       # reviews per API call — tune down to 5 if still getting 429s
DELAY = 15            # seconds between calls — tune up if needed (60 / your RPM limit)

# ── Updated prompt: handles a numbered batch of reviews ──────────────────────
SYSTEM_PROMPT = """You are a hotel review analyzer. You will receive multiple reviews numbered [1], [2], etc.

For EACH review, extract signals for these 5 attributes:
1. cleanliness - how clean the rooms, bathrooms, common areas are
2. staff_service - quality of staff, helpfulness, friendliness
3. wifi_quality - internet/wifi speed and reliability
4. location_accessibility - proximity to public transit
5. safety_&_securiy - safety and security for room lock and guard and luggage safety

For each attribute return:
- sentiment: "positive", "negative", "neutral", or "not_mentioned"
- confidence: 0.0 to 1.0
- evidence: exact sentence(s) from the review (empty string if not_mentioned)

Return ONLY a valid JSON array, one object per review, no explanation, no markdown:
[
  {
    "cleanliness": {"sentiment": "...", "confidence": 0.0, "evidence": "..."},
    "staff_service": {"sentiment": "...", "confidence": 0.0, "evidence": "..."},
    "wifi_quality": {"sentiment": "...", "confidence": 0.0, "evidence": "..."},
    "location_accessibility": {"sentiment": "...", "confidence": 0.0, "evidence": "..."},
    "safety_&_securiy": {"sentiment": "...", "confidence": 0.0, "evidence": "..."}
  },
  ...
]"""


def get_default_result():
    return {attr: {"sentiment": "not_mentioned", "confidence": 0.0, "evidence": ""} for attr in ATTRIBUTES}


def extract_batch(reviews: list[str], max_retries=2) -> list[dict]:
    """Send a batch of reviews in one API call."""
    # formate will be :- [index]{review}
    batch_text = "\n\n".join(
        f"[{i+1}] {str(r)[:800]}" for i, r in enumerate(reviews)
    )

    for attempt in range(max_retries):
        try:
            response = client.chat.complete(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": batch_text}
                ],
                temperature=0,  # for discriminatry responce no creative answer
                max_tokens=400 * len(reviews)   # scale tokens with batch size
            )

            raw = response.choices[0].message.content.strip()
            raw = re.sub(r'```json|```', '', raw).strip()
            results = json.loads(raw)

            # Validate we got the right number of results back
            if isinstance(results, list) and len(results) == len(reviews):
                return results
            else:
                print(f"\nWarning: expected {len(reviews)} results, got {len(results)}. Padding with defaults.")
                # Pad missing results with defaults
                while len(results) < len(reviews):
                    results.append(get_default_result())
                return results[:len(reviews)]

        except json.JSONDecodeError as e:
            print(f"\nJSON parse error (attempt {attempt+1}): {e}")
            if attempt == max_retries - 1:
                return [get_default_result()] * len(reviews)

        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                wait = 30 * (2 ** attempt)   # 30s, 60s, 120s, 240s, 480s
                print(f"\nRate limited. Waiting {wait}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(wait)
            else:
                print(f"\nAPI error (attempt {attempt+1}): {e}")
                time.sleep(5)

            if attempt == max_retries - 1:
                return [get_default_result()] * len(reviews)

    return [get_default_result()] * len(reviews)


def load_checkpoint(path="checkpoint.json"):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return {}


def save_checkpoint(data, path="checkpoint.json"):
    with open(path, "w") as f:
        json.dump(data, f)


# ── Main ─────────────────────────────────────────────────────────────────────

checkpoint = load_checkpoint()
already_done = set(str(k) for k in checkpoint.keys())
results = list(checkpoint.values())

print(f"Resuming: {len(already_done)} done, {len(balanced_df) - len(already_done)} remaining.")

# Split remaining rows into batches
remaining = [(idx, row) for idx, row in balanced_df.iterrows() if str(idx) not in already_done]
batches = [remaining[i:i+BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]

print(f"Processing {len(remaining)} reviews in {len(batches)} batches of {BATCH_SIZE}...")

for batch in tqdm(batches):
    indices = [idx for idx, _ in batch]
    reviews = [row['Review'] for _, row in batch]
    rows    = [row for _, row in batch]

    extractions = extract_batch(reviews)

    for (idx, row), extraction in zip(batch, extractions):
        record = {
            'review_idx': idx,
            'rating': row['Rating'],
            'review_text': row['Review'][:300],
            **{f"{attr}_sentiment": extraction.get(attr, {}).get('sentiment', 'not_mentioned') for attr in ATTRIBUTES},
            **{f"{attr}_confidence": extraction.get(attr, {}).get('confidence', 0.0) for attr in ATTRIBUTES},
            **{f"{attr}_evidence": extraction.get(attr, {}).get('evidence', '') for attr in ATTRIBUTES},
        }
        results.append(record)
        checkpoint[str(idx)] = record

    save_checkpoint(checkpoint)
    pd.DataFrame(results).to_csv('labeled_reviews.csv', index=False)
    time.sleep(DELAY)   # one sleep per BATCH, not per review

print(f"\nDone! {len(results)} reviews saved to labeled_reviews.csv")
pd.DataFrame(results).head(3)

Resuming: 0 done, 750 remaining.
Processing 750 reviews in 75 batches of 10...


 53%|█████▎    | 40/75 [27:24<23:12, 39.80s/it]

100%|██████████| 75/75 [51:54<00:00, 41.53s/it]


Done! 750 reviews saved to labeled_reviews.csv


,review_idx,rating,review_text,cleanliness_sentiment,staff_service_sentiment,wifi_quality_sentiment,location_accessibility_sentiment,safety_&_securiy_sentiment,cleanliness_confidence,staff_service_confidence,wifi_quality_confidence,location_accessibility_confidence,safety_&_securiy_confidence,cleanliness_evidence,staff_service_evidence,wifi_quality_evidence,location_accessibility_evidence,safety_&_securiy_evidence
0,18754,4,excellent allround traffic noise junior hotel ...,positive,positive,not_mentioned,positive,not_mentioned,0.95,0.95,0.0,0.95,0.0,rooms clean simply tastefully furnished air co...,staff way helpful reception manned 24 hours day,,situated avinguda sarria 5 minutes walk hospit...,
1,2440,4,"magnificent trip vieques, love staying somewha...",neutral,positive,not_mentioned,not_mentioned,not_mentioned,0.70,0.95,0.0,0.00,0.0,buildings apartments amazing,kris kurt great helpful,,,
2,16540,4,tried tested formula aman really got pat fabul...,positive,positive,not_mentioned,not_mentioned,not_mentioned,0.95,1.00,0.0,0.00,0.0,beautiful resort amazing design despite 15 yea...,"it__Ç_é_ definitely staff makes place, persona...",,,


# **Training labeled data**

Train a lightweight multi-label classifier per attribute using TF-IDF + Logistic Regression.
Why this stack:
  - Fast to train (seconds, not hours)
  - Interpretable
  -Small data regime (< 5K samples)
  -Nature of your data (short, explicit text
  eg.{"clean room", "dirty bathroom", "slow wifi"})

# *trained on different atributes["cleanliness", "staff_service", "wifi_quality", "location_accessibility", "safety_&_securiy"]seperately*



In [16]:

import pandas as pd
import numpy as np
import json
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

# ── Load labeled data ────────────────────────────────────────────────────────
labeled = pd.read_csv("labeled_reviews.csv")
print(f"Loaded {len(labeled)} labeled reviews")
print(labeled.head(2))

ATTRIBUTES = ["cleanliness", "staff_service", "wifi_quality", "location_accessibility", "safety_&_securiy"]
SENTIMENTS = ["positive", "negative", "neutral", "not_mentioned"]


from sklearn.model_selection import train_test_split

train_labeled, test_labeled = train_test_split(labeled, test_size=0.2, random_state=42, stratify=labeled["rating"])

print(f"\nTrain test length of data: {len(train_labeled)} | Test: {len(test_labeled)}")


# ── Train one pipeline per attribute ───────────
models = {}
results_summary = {}

for attr in ATTRIBUTES:
    # target column to be sentiment "positive","negative","neutral,"not mentioned"
    target_col = f"{attr}_sentiment"

    # Drop rows where confidence is very low (label quality filter)
    conf_col = f"{attr}_confidence"
    subset = train_labeled[train_labeled[conf_col] >= 0.5].copy()

    #subset is all confident review labeled
    #removing NAN to empty string for tfidf
    X_train = subset["review_text"].fillna("")

    #replacing target Nan to not mentioned as empty would be not in target
    y_train = subset[target_col].fillna("not_mentioned")

    X_test = test_labeled["review_text"].fillna("")
    y_test = test_labeled[target_col].fillna("not_mentioned")

    # Class balance report
    dist = y_train.value_counts()
    print(f"\n── {attr} ──")
    print(dist)

    # Handle severe class imbalance with class_weight='balanced'
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),     # unigrams + bigrams capture "not clean", "great wifi"
            max_features=15000,
            min_df=2,
            sublinear_tf=True,      # log scaling dampens high-frequency terms
            strip_accents="unicode"
        )),
        ("clf", LogisticRegression(
            max_iter=500,
            class_weight="balanced",  # compensates for "not_mentioned" dominance
            C=1.0,
            solver="lbfgs",
            multi_class="multinomial"
        ))
    ])

    pipeline.fit(X_train, y_train)

    # Evaluate
    y_pred = pipeline.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    print(f"  Macro F1: {macro_f1:.3f}")
    print(classification_report(y_test, y_pred, zero_division=0))

    models[attr] = pipeline
    results_summary[attr] = {
        "macro_f1": round(macro_f1, 3),
        "per_class": {
            cls: {
                "f1": round(report[cls]["f1-score"], 3),
                "precision": round(report[cls]["precision"], 3),
                "recall": round(report[cls]["recall"], 3),
                "support": report[cls]["support"]
            }
            for cls in SENTIMENTS if cls in report
        }
    }

    # Save model
    joblib.dump(pipeline, f"model_{attr}.joblib")

# Save results summary for writeup
with open("eval_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)

print("\n\nAll models saved. Evaluation summary:")
for attr, res in results_summary.items():
    print(f"  {attr}: Macro F1 = {res['macro_f1']}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Loaded 750 labeled reviews
   review_idx  rating                                        review_text  \
0       18754       4  excellent allround traffic noise junior hotel ...   
1        2440       4  magnificent trip vieques, love staying somewha...   

  cleanliness_sentiment staff_service_sentiment wifi_quality_sentiment  \
0              positive                positive          not_mentioned   
1               neutral                positive          not_mentioned   

  location_accessibility_sentiment safety_&_securiy_sentiment  \
0                         positive              not_mentioned   
1                    not_mentioned              not_mentioned   

   cleanliness_confidence  staff_service_confidence  wifi_quality_confidence  \
0                    0.95                      0.95                      0.0   
1                    0.70                      0.95                      0.0   

   location_accessibility_confidence  safety_&_securiy_confidence  \
0              

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


  Macro F1: 0.345
               precision    recall  f1-score   support

        mixed       0.00      0.00      0.00         1
     negative       0.61      0.93      0.74        44
      neutral       0.33      0.07      0.12        14
not_mentioned       1.00      0.05      0.10        19
     positive       0.73      0.81      0.77        72

     accuracy                           0.67       150
    macro avg       0.54      0.37      0.34       150
 weighted avg       0.69      0.67      0.61       150


── wifi_quality ──
wifi_quality_sentiment
not_mentioned    130
negative          25
positive          14
neutral            5
Name: count, dtype: int64
  Macro F1: 0.375
               precision    recall  f1-score   support

     negative       0.29      0.33      0.31         6
      neutral       0.00      0.00      0.00         1
not_mentioned       0.93      0.96      0.94       137
     positive       0.50      0.17      0.25         6

     accuracy                       

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



── location_accessibility ──
location_accessibility_sentiment
positive         263
not_mentioned     68
neutral           52
negative          35
Name: count, dtype: int64
  Macro F1: 0.324
               precision    recall  f1-score   support

     negative       0.00      0.00      0.00         8
      neutral       0.00      0.00      0.00         8
not_mentioned       0.74      0.48      0.58        71
     positive       0.61      0.86      0.72        63

     accuracy                           0.59       150
    macro avg       0.34      0.33      0.32       150
 weighted avg       0.61      0.59      0.58       150


── safety_&_securiy ──
safety_&_securiy_sentiment
negative         96
not_mentioned    94
positive         26
neutral          22
Name: count, dtype: int64
  Macro F1: 0.387
               precision    recall  f1-score   support

     negative       0.22      0.57      0.32        23
      neutral       0.33      0.20      0.25         5
not_mentioned       0.84 

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


# **AGGREGATION -> TIERS + EVIDENCE**

Tier logic (per attribute, based on % positive reviews mentioning the attribute):
  Elite  --- ≥ 80% positive (of those mentioning it), ≥ 5 reviews mentioning it
  Superior --> 60–79% positive
  Premium  --> 40–59% positive  (or: more negative than positive but not strongly so)
  Fail    ---> < 40% positive (of mentions), or ≥ 60% negative

Low-evidence handling:

  If a hotel has < 3
  reviews mentioning an attribute--> tier = "insufficient_data"


  If total reviews < 5 --> flag as low_confidence

In [17]:

import pandas as pd
import numpy as np
import joblib
import json
from collections import defaultdict

ATTRIBUTES = ["cleanliness", "staff_service", "wifi_quality", "location_accessibility", "safety_&_securiy"]

# ── Load labeled data + models ───────────────────────────────────────────────
labeled = pd.read_csv("labeled_reviews.csv")
models = {attr: joblib.load(f"model_{attr}.joblib") for attr in ATTRIBUTES}


# ── Inference function: predict on new reviews ───────────────────────────────
def predict_review(review_text: str) -> dict:
    """Return sentiment + confidence for each attribute."""
    result = {}
    for attr, model in models.items():
        proba = model.predict_proba([review_text])[0]
        classes = model.classes_
        pred_class = classes[np.argmax(proba)]
        confidence = float(np.max(proba))
        result[attr] = {"sentiment": pred_class, "confidence": confidence}
    return result


# ── Tier mapping function ────────────────────────────────────────────────────
MIN_MENTIONS = 3   # minimum reviews mentioning attribute for a valid tier

def compute_tier(mention_sentiments: list[str]) -> str:
    """
    mention_sentiments: list of sentiments from reviews that actually mention this attribute
    (i.e., already filtered to exclude 'not_mentioned')
    """
    n = len(mention_sentiments)
    if n < MIN_MENTIONS:
        return "insufficient_data"

    counts = pd.Series(mention_sentiments).value_counts()
    pos = counts.get("positive", 0)
    neg = counts.get("negative", 0)
    neu = counts.get("neutral", 0)
    total = pos + neg + neu  # total that mentioned it

    if total == 0:
        return "insufficient_data"

    pct_positive = pos / total
    pct_negative = neg / total

    # Tier thresholds — defensible via distribution of training data
    # Elite: strong positive signal
    # Fail: strong negative signal (or reversed: pct_negative dominates)

    if pct_positive >= 0.80:
        return "Elite"
    elif pct_positive >= 0.60:
        return "Superior"
    elif pct_negative >= 0.60:
        return "Fail"
    else:
        return "Premium"


# ── Simulate hotel-level aggregation ────────────────────────────────────────
# NOTE: The TripAdvisor dataset doesn't have hotel IDs —
# we'll simulate by grouping on 'review_idx % 50' (50 mock hotels).
# In production: group by hotel_id column.

labeled["mock_hotel_id"] = labeled["review_idx"] % 50

hotel_tiers = {}
hotel_evidence = defaultdict(dict)
hotel_meta = {}

for hotel_id, group in labeled.groupby("mock_hotel_id"):
    n_reviews = len(group)
    hotel_meta[hotel_id] = {
        "n_reviews": n_reviews,
        "low_confidence": n_reviews < 5,
        "avg_rating": round(group["rating"].mean(), 2)
    }

    tiers = {}
    for attr in ATTRIBUTES:
        sent_col = f"{attr}_sentiment"
        conf_col = f"{attr}_confidence"
        evid_col = f"{attr}_evidence"

        # Only use predictions with confidence ≥ 0.4
        high_conf = group[group[conf_col] >= 0.4]

        # Filter to reviews that actually mention this attribute
        mentioned = high_conf[high_conf[sent_col] != "not_mentioned"]

        mention_sentiments = mentioned[sent_col].tolist()
        tier = compute_tier(mention_sentiments)
        tiers[attr] = tier

        # Evidence: top 3 sentences driving the tier
        if tier in ("Elite", "Superior"):
            evidence_pool = mentioned[mentioned[sent_col] == "positive"]
        elif tier == "Fail":
            evidence_pool = mentioned[mentioned[sent_col] == "negative"]
        else:
            evidence_pool = mentioned

        # Sort by confidence descending, take top 3
        top_evidence = (
            evidence_pool
            .sort_values(conf_col, ascending=False)
            .head(3)[evid_col]
            .dropna()
            .tolist()
        )

        hotel_evidence[hotel_id][attr] = {
            "tier": tier,
            "n_mentions": len(mentioned),
            "sentiment_breakdown": {
                "positive": int((mentioned[sent_col] == "positive").sum()),
                "negative": int((mentioned[sent_col] == "negative").sum()),
                "neutral":  int((mentioned[sent_col] == "neutral").sum()),
            },
            "evidence_sentences": top_evidence
        }

    hotel_tiers[hotel_id] = tiers

# ── Save outputs ─────────────────────────────────────────────────────────────
tier_rows = []
for hotel_id, tiers in hotel_tiers.items():
    row = {"hotel_id": hotel_id, **hotel_meta[hotel_id], **tiers}
    tier_rows.append(row)

tier_df = pd.DataFrame(tier_rows)
tier_df.to_csv("hotel_tiers.csv", index=False)
print("hotel_tiers.csv saved")

with open("hotel_evidence.json", "w") as f:
    json.dump({str(k): v for k, v in hotel_evidence.items()}, f, indent=2)
print("hotel_evidence.json saved")

# ── Quick summary ─────────────────────────────────────────────────────────────
print("\n── Tier distribution per attribute ──")
for attr in ATTRIBUTES:
    dist = tier_df[attr].value_counts()
    print(f"\n{attr}:")
    print(dist)

print("\n── Sample hotel evidence ──")
sample = hotel_evidence.get(0, {})
for attr, info in sample.items():
    print(f"\n{attr}: {info['tier']}")
    for s in info['evidence_sentences'][:1]:
        print(f"  > {s[:120]}...")

hotel_tiers.csv saved
hotel_evidence.json saved

── Tier distribution per attribute ──

cleanliness:
cleanliness
Premium     28
Superior    12
Elite        6
Fail         4
Name: count, dtype: int64

staff_service:
staff_service
Premium     33
Superior    12
Elite        3
Fail         2
Name: count, dtype: int64

wifi_quality:
wifi_quality
insufficient_data    45
Premium               3
Fail                  2
Name: count, dtype: int64

location_accessibility:
location_accessibility
Elite                25
Superior             17
Premium               7
insufficient_data     1
Name: count, dtype: int64

safety_&_securiy:
safety_&_securiy
Fail                 29
insufficient_data    12
Premium               6
Superior              3
Name: count, dtype: int64

── Sample hotel evidence ──

cleanliness: Elite
  > it spotless....

staff_service: Elite
  > must make special mention lads mikes coffee shopand alexander lobby bar 24 hour smiles entertainment 90 staff maids mana...

wifi_qualit